<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/GenerativeAI/Notebooks/Deployment/Ollama/embedding_qwen3_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embedding Model + LLM Deployment using Ollama (Qwen3)

This notebook runs an embedding model alongside an LLM using Ollama — showing a complete RAG pipeline end to end.

**Models:**
- Embedding: `nomic-embed-text` — open source from Nomic AI, 768-dimensional vectors
- LLM: `qwen3:0.6b` — Alibaba's Qwen3, 0.6B parameters, latest generation, fast on T4

Qwen3 is used here because it's lightweight, loads reliably on Colab's free T4 GPU, and performs well for its size. The RAG pipeline works the same regardless of which LLM you plug in.

> Set runtime to **T4 GPU** before running.

## Installing Ollama

`zstd` is needed by the Ollama installer to unpack itself — install it first or the setup fails partway through.

In [1]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Starting the Server

Ollama runs as a local server on port `11434`. We kick it off in the background with `subprocess.Popen` so the notebook doesn't hang, then wait until it's up.

In [2]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


## Pulling the Models

We pull both models upfront. Ollama manages them through the same server — just switch model names in each request.

In [3]:
# embedding model
!ollama pull nomic-embed-text

# LLM
!ollama pull qwen3:0.6b

## Generating Embeddings

Ollama exposes a `/api/embeddings` endpoint. Send text, get back a 768-dimensional vector representing the meaning of that text.

In [4]:
import requests, json

def get_embedding(text):
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text}
    )
    return response.json()["embedding"]

embedding = get_embedding("What is a large language model?")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimensions: 768
First 5 values: [-1.1624970436096191, 1.9516496658325195, -3.4593796730041504, -1.9797314405441284, 0.10939395427703857]


## Semantic Similarity

Comparing sentences using cosine similarity — a score of 1.0 means identical meaning, 0.0 means completely unrelated. The two AI sentences should score much higher than the pizza comparison even though none share exact words.

In [5]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "Large language models are trained on massive text datasets.",
    "Neural networks learn patterns from large amounts of data.",
    "Pizza is a popular dish made with dough, sauce, and cheese."
]

embeddings = [get_embedding(s) for s in sentences]

print("Similarity: AI sentence 1 vs AI sentence 2:", round(cosine_similarity(embeddings[0], embeddings[1]), 4))
print("Similarity: AI sentence 1 vs Pizza sentence:", round(cosine_similarity(embeddings[0], embeddings[2]), 4))

Similarity: AI sentence 1 vs AI sentence 2: 0.6983
Similarity: AI sentence 1 vs Pizza sentence: 0.4287


## RAG Pipeline

RAG in four steps:
1. Embed a set of documents
2. Embed the incoming question
3. Find the most similar document using cosine similarity
4. Pass that document as context to the LLM

Production systems do the same thing at scale with a vector database. Here we keep it minimal so the logic is easy to follow.

In [6]:
from openai import OpenAI

!pip install openai -q

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

documents = [
    "Ollama is a tool for running large language models locally on your machine without an internet connection.",
    "Qwen3 is an open-source language model developed by Alibaba, available in sizes from 0.6B to 235B parameters.",
    "Gemma is an open-source language model created by Google DeepMind, available in 2B and 7B parameter sizes.",
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents."
]

doc_embeddings = [get_embedding(doc) for doc in documents]

def rag_answer(question):
    q_embedding = get_embedding(question)
    similarities = [cosine_similarity(q_embedding, doc_emb) for doc_emb in doc_embeddings]
    best_idx = int(np.argmax(similarities))
    best_doc = documents[best_idx]

    print(f"Most relevant document (similarity={similarities[best_idx]:.4f}):")
    print(f"  -> {best_doc}\n")

    prompt = f"Use the following context to answer the question.\n\nContext: {best_doc}\n\nQuestion: {question}"
    response = client.chat.completions.create(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    return response.choices[0].message.content

answer = rag_answer("What is Qwen3 and who made it?")
print("Answer:", answer)

Most relevant document (similarity=0.7632):
  -> Qwen3 is an open-source language model developed by Alibaba, available in sizes from 0.6B to 235B parameters.

Answer: Qwen3 is an open-source language model developed by Alibaba. It offers a range of model sizes from 0.6B to 235B parameters, making it suitable for various applications.


## Streaming with the LLM

Tokens come back as they're generated — same as in the other Ollama notebooks.

In [7]:
stream = client.chat.completions.create(
    model="qwen3:0.6b",
    messages=[{"role": "user", "content": "Explain what RAG is in simple terms."}],
    max_tokens=200,
    stream=True
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

## Batch Embedding

Embedding multiple documents in a loop — the pattern you'd use before indexing them into a vector store.

In [8]:
texts = [
    "Ollama makes local LLM deployment easy.",
    "Qwen3 is one of the best small open-source models.",
    "Embeddings capture semantic meaning of text.",
    "RAG improves LLM answers using retrieved context."
]

batch_embeddings = [get_embedding(t) for t in texts]

for text, emb in zip(texts, batch_embeddings):
    print(f"'{text[:50]}' -> vector of length {len(emb)}")

'Ollama makes local LLM deployment easy.' -> vector of length 768
'Qwen3 is one of the best small open-source models.' -> vector of length 768
'Embeddings capture semantic meaning of text.' -> vector of length 768
'RAG improves LLM answers using retrieved context.' -> vector of length 768


## Summary

| | |
|---|---|
| Embedding model | `nomic-embed-text` via Ollama |
| LLM | `qwen3:0.6b` via Ollama |
| Embedding endpoint | `POST /api/embeddings` |
| LLM endpoint | `POST /v1/chat/completions` (OpenAI-compatible) |
| What we built | semantic similarity + RAG pipeline |

One Ollama server handles both models. No extra infrastructure, no API keys, just pull and run — that's the main advantage of Ollama for local development and prototyping.